In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

In [3]:
import pandas as pd
import os


DATASET_PATH = r'/content/drive/MyDrive/UCI HAR Dataset/'


# 1. 피처(Feature) 이름 불러오기
# 'features.txt' 파일에는 각 데이터(X_train, X_test)의 561개 열이 무엇을 의미하는지 적혀있습니다.
feature_name_df = pd.read_csv(os.path.join(DATASET_PATH, 'features.txt'),
                              sep='\s+', header=None, names=['index', 'feature_name'])

# 피처 이름을 리스트로 만듭니다. (나중에 DataFrame의 열 이름으로 사용)
feature_names = feature_name_df['feature_name'].tolist()


# 2. 활동(Activity) 레이블 이름 불러오기
# 'activity_labels.txt' 파일에는 숫자(1, 2, 3...)가 어떤 활동(WALKING, SITTING...)인지 적혀있습니다.
activity_labels_df = pd.read_csv(os.path.join(DATASET_PATH, 'activity_labels.txt'),
                                 sep='\s+', header=None, names=['activity_id', 'activity_name'])

print('--- 활동 레이블 ---')
print(activity_labels_df)
print('---------------------\n')


# 3. 훈련(train) 데이터 불러오기
# -----------------------------
# 'X_train.txt': 훈련용 피처(센서 값) 데이터
# 'y_train.txt': 훈련용 레이블(활동 번호) 데이터

# (1) X_train (피처 데이터) 불러오기
X_train_df = pd.read_csv(os.path.join(DATASET_PATH, 'train/X_train.txt'),
                         sep='\s+', header=None)
# 위에서 불러온 feature_names를 열 이름으로 설정
X_train_df.columns = feature_names

# (2) y_train (레이블 데이터) 불러오기
y_train_df = pd.read_csv(os.path.join(DATASET_PATH, 'train/y_train.txt'),
                         header=None, names=['activity_id'])


# 4. 테스트(test) 데이터 불러오기
# -----------------------------
# 'X_test.txt': 테스트용 피처(센서 값) 데이터
# 'y_test.txt': 테스트용 레이블(활동 번호) 데이터

# (1) X_test (피처 데이터) 불러오기
X_test_df = pd.read_csv(os.path.join(DATASET_PATH, 'test/X_test.txt'),
                        sep='\s+', header=None)
# 훈련 데이터와 동일하게 열 이름 설정
X_test_df.columns = feature_names

# (2) y_test (레이블 데이터) 불러오기
y_test_df = pd.read_csv(os.path.join(DATASET_PATH, 'test/y_test.txt'),
                        header=None, names=['activity_id'])


# 5. 데이터 로드 확인
print('--- 훈련 데이터 (train) ---')
print(f'X_train (피처) shape: {X_train_df.shape}')
print(f'y_train (레이블) shape: {y_train_df.shape}')
print('\n--- y_train 레이블 분포 ---')
print(y_train_df['activity_id'].value_counts()) # 레이블이 골고루 있는지 확인

print('\n--- 테스트 데이터 (test) ---')
print(f'X_test (피처) shape: {X_test_df.shape}')
print(f'y_test (레이블) shape: {y_test_df.shape}')

print('\n--- X_train 데이터 샘플 (상위 3개) ---')
print(X_train_df.head(3))

<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:34: SyntaxWarning: invalid escape sequence '\s'
<>:50: SyntaxWarning: invalid escape sequence '\s'
<>:11: SyntaxWarning: invalid escape sequence '\s'
<>:20: SyntaxWarning: invalid escape sequence '\s'
<>:34: SyntaxWarning: invalid escape sequence '\s'
<>:50: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-853624603.py:11: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+', header=None, names=['index', 'feature_name'])
/tmp/ipython-input-853624603.py:20: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+', header=None, names=['activity_id', 'activity_name'])
/tmp/ipython-input-853624603.py:34: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+', header=None)
/tmp/ipython-input-853624603.py:50: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+', header=None)


--- 활동 레이블 ---
   activity_id       activity_name
0            1             WALKING
1            2    WALKING_UPSTAIRS
2            3  WALKING_DOWNSTAIRS
3            4             SITTING
4            5            STANDING
5            6              LAYING
---------------------

--- 훈련 데이터 (train) ---
X_train (피처) shape: (7352, 561)
y_train (레이블) shape: (7352, 1)

--- y_train 레이블 분포 ---
activity_id
6    1407
5    1374
4    1286
1    1226
2    1073
3     986
Name: count, dtype: int64

--- 테스트 데이터 (test) ---
X_test (피처) shape: (2947, 561)
y_test (레이블) shape: (2947, 1)

--- X_train 데이터 샘플 (상위 3개) ---
   tBodyAcc-mean()-X  tBodyAcc-mean()-Y  tBodyAcc-mean()-Z  tBodyAcc-std()-X  \
0           0.288585          -0.020294          -0.132905         -0.995279   
1           0.278419          -0.016411          -0.123520         -0.998245   
2           0.279653          -0.019467          -0.113462         -0.995380   

   tBodyAcc-std()-Y  tBodyAcc-std()-Z  tBodyAcc-mad()-X  tBodyAcc-mad()

In [4]:
import numpy as np
from tensorflow.keras.utils import to_categorical # 원-핫 인코딩을 위한 유틸리티

# --- X (피처) 데이터 3D로 변환 ---
# .values를 사용해 DataFrame을 Numpy 배열로 변환
X_train_np = X_train_df.values
X_test_np = X_test_df.values

# (샘플 수, 561) -> (샘플 수, 561, 1) 형태로 reshape
X_train_cnn = X_train_np.reshape(X_train_np.shape[0], X_train_np.shape[1], 1)
X_test_cnn = X_test_np.reshape(X_test_np.shape[0], X_test_np.shape[1], 1)

print('--- X 데이터 형태 변환 (CNN 입력용) ---')
print(f'원본 X_train shape: {X_train_np.shape}')
print(f'변환 X_train shape: {X_train_cnn.shape}')


# --- y (레이블) 데이터 원-핫 인코딩 ---
# 1. 레이블이 1~6이므로, 0~5 범위로 변경 (중요!)
y_train_np = y_train_df['activity_id'] - 1
y_test_np = y_test_df['activity_id'] - 1

# 2. 0~5 범위의 숫자를 원-핫 인코딩
# (총 6개 클래스)
y_train_onehot = to_categorical(y_train_np, num_classes=6)
y_test_onehot = to_categorical(y_test_np, num_classes=6)

print('\n--- y 데이터 원-핫 인코딩 (CNN 출력용) ---')
print(f'원본 y_train shape: {y_train_np.shape}')
print(f'변환 y_train shape: {y_train_onehot.shape}')
print('\n샘플 확인:')
print(f'  원본 레이블 (5): {y_train_np[0]}')
print(f'  원-핫 인코딩 (5 -> [0,0,0,0,1,0]): {y_train_onehot[0]}')

--- X 데이터 형태 변환 (CNN 입력용) ---
원본 X_train shape: (7352, 561)
변환 X_train shape: (7352, 561, 1)

--- y 데이터 원-핫 인코딩 (CNN 출력용) ---
원본 y_train shape: (7352,)
변환 y_train shape: (7352, 6)

샘플 확인:
  원본 레이블 (5): 4
  원-핫 인코딩 (5 -> [0,0,0,0,1,0]): [0. 0. 0. 0. 1. 0.]


In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

In [6]:
# --- 1. CNN 모델 구성하기 ---
# (561, 1) 형태의 입력을 받는 1D CNN 모델을 만듭니다.

model = Sequential()

# 첫 번째 Convolution + Pooling 레이어
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(561, 1)))
model.add(MaxPooling1D(pool_size=2))

# 두 번째 Convolution + Pooling 레이어 (선택 사항이지만 성능 향상에 도움)
model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))

# 과적합을 방지하기 위한 Dropout
model.add(Dropout(0.5))

# 3D 데이터를 1D로 평평하게 펴주기
model.add(Flatten())

# Dense 레이어 (신경망)
model.add(Dense(100, activation='relu'))

# 출력 레이어 (6개 클래스를 분류하므로 Dense(6), 활성화 함수는 softmax)
model.add(Dense(6, activation='softmax'))

# 모델 구조 요약
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 559, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 279, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 277, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 138, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 138, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 17664)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │     1,766,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           606 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,792,066 (6.84 MB)

 Trainable params: 1,792,066 (6.84 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# --- 2. 모델 컴파일하기 ---
# 모델을 학습할 수 있도록 설정합니다.
model.compile(optimizer='adam',                   # 옵티마이저 (가장 무난함)
              loss='categorical_crossentropy',    # 손실 함수 (원-핫 인코딩 시 사용)
              metrics=['accuracy'])               # 평가 지표 (정확도)

In [10]:
# --- 3. 모델 학습 (Training) ---
print("\n--- 모델 학습 시작 ---")


history = model.fit(X_train_cnn, y_train_onehot,
                    epochs=30,
                    batch_size=64,
                    validation_data=(X_test_cnn, y_test_onehot),
                    verbose=1)

print("--- 모델 학습 완료 ---")


--- 모델 학습 시작 ---
Epoch 1/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9915 - loss: 0.0244 - val_accuracy: 0.9338 - val_loss: 0.2512
Epoch 2/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9916 - loss: 0.0262 - val_accuracy: 0.9613 - val_loss: 0.1550
Epoch 3/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9955 - loss: 0.0121 - val_accuracy: 0.9542 - val_loss: 0.1664
Epoch 4/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9938 - loss: 0.0200 - val_accuracy: 0.9579 - val_loss: 0.1682
Epoch 5/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9902 - loss: 0.0228 - val_accuracy: 0.9644 - val_loss: 0.1480
Epoch 6/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9934 - loss: 0.0150 - val_accuracy: 0.9610 - val_loss: 0.1584
Epoch 7/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9935 - loss: 0.0169 - val_accuracy: 0.9627 - val_loss: 0.1388
Epoch 8/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9938 - loss: 0.01

In [11]:
# --- 4. 모델 평가 (Evaluation) ---
# 학습이 끝난 모델의 최종 성능을 테스트 데이터로 평가합니다.
print("\n--- 모델 평가 (Test Set) ---")
loss, accuracy = model.evaluate(X_test_cnn, y_test_onehot)

print(f'\nTest Loss (손실): {loss:.4f}')
print(f'Test Accuracy (정확도): {accuracy:.4f}')


--- 모델 평가 (Test Set) ---
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9604 - loss: 0.1738

Test Loss (손실): 0.1777
Test Accuracy (정확도): 0.9617
